# ADNI_PrimeKGL
Copied from the original project. Paths are relative to the main folder.
Full PrimeKG embeddings and entity indices are shared by all four cohorts.


In [ ]:
# Paths relative to the main folder; portable between Windows and Linux.
from pathlib import Path
import os
import sys
_starts = [Path.cwd(), Path(globals().get("__file__", Path.cwd() / "notebook")).resolve().parent]
CLINICALKG_ROOT = next(p for start in _starts for p in (start, *start.parents)
                      if (p / "full_primekg.py").is_file() and (p / "paths.py").is_file())
for _path in (CLINICALKG_ROOT.parent, CLINICALKG_ROOT):
    if str(_path) in sys.path:
        sys.path.remove(str(_path))
    sys.path.insert(0, str(_path))
from clinicalKG.paths import PROJECT_ROOT, SHARED_PRIMEKG, raw_primekg_path, work_dir, asset_path
DATASET = 'ADNI'
METHOD = 'PrimeKGL'
WORK_DIR = work_dir(DATASET, METHOD)
DATASET_DIR = PROJECT_ROOT / DATASET
if METHOD == "PrimeKGL":
    SHARED_PRIMEKG.mkdir(parents=True, exist_ok=True)
# A notebook kernel may previously have imported the original project's modules.
for _name in list(sys.modules):
    if _name == "experiments" or _name.startswith("experiments."):
        _file = getattr(sys.modules[_name], "__file__", None)
        if _file and not Path(_file).resolve().is_relative_to(CLINICALKG_ROOT):
            del sys.modules[_name]
if "model1" in sys.modules:
    _file = getattr(sys.modules["model1"], "__file__", None)
    if _file and Path(_file).resolve().parent != WORK_DIR:
        del sys.modules["model1"]
if str(WORK_DIR) in sys.path:
    sys.path.remove(str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR))
if DATASET_DIR.is_dir():
    os.chdir(DATASET_DIR)
os.environ.setdefault("KG_ABLATION_OUTPUT", str(CLINICALKG_ROOT / "results"))


In [ ]:
# Full PrimeKG is used directly: no seed extraction, blacklist, or Top-k.
RAW_KG_PATH = str(raw_primekg_path())
SHARED_PRIMEKG.mkdir(parents=True, exist_ok=True)
if not Path(RAW_KG_PATH).is_file():
    raise FileNotFoundError(RAW_KG_PATH)
print("Full PrimeKG source:", RAW_KG_PATH)
print("All cohorts share graph-trained assets in:", SHARED_PRIMEKG)


In [ ]:
import pandas as pd
import json
import numpy as np
import warnings

warnings.filterwarnings('ignore')

ADNI_DATA_PATH = str(DATASET_DIR / 'ADNI诊断.csv')
SEED_CSV_PATH = str(WORK_DIR / 'ADNI_Seed_DementiaHKG.csv')
REFINED_KG_PATH = str(raw_primekg_path())
OUTPUT_GRAPH_PATH = str(SHARED_PRIMEKG / 'PrimeKG-G.csv')
OUTPUT_DICT_PATH = str(WORK_DIR / 'ADNI-PrimeKG-Raw-Entities.json')

MAX_SEQ_LEN = 35
PAD_TOKEN = "[PAD]"

def build_patient_mapping():
    print("==================================================")
    print("🔗 阶段 3: 构建患者动态映射与客观图谱格式化 (已引入序列填充与注意力掩码)...")

    try:
        kg_df = pd.read_csv(REFINED_KG_PATH)
        triplets_df = kg_df[['x_name', 'relation', 'y_name']].copy()
        triplets_df.columns = ['head', 'relation', 'tail']
        
        triplets_df.to_csv(OUTPUT_GRAPH_PATH, index=False, encoding='utf-8')
        print(f"   ✅ 标准化图谱已生成: {OUTPUT_GRAPH_PATH} (共 {len(triplets_df)} 条边)")
        print("   🔒 [安全检查]: 该图谱中不存在任何患者节点，已彻底阻断 Transductive 泄露。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {REFINED_KG_PATH}，请确保上一阶段已成功运行。")
        return

    seed_df = pd.read_csv(SEED_CSV_PATH)
    mapping_df = seed_df[seed_df['Category'] == 'Clinical_Mapping']
    adni_to_pkg_map = dict(zip(mapping_df['ADNI_Feature'], mapping_df['PrimeKG_Node']))
    print(f"\n   📊 成功加载 {len(adni_to_pkg_map)} 条临床特征映射规则。")

    threshold_rules = {
        "mmse": lambda x: x <= 24,
        "moca": lambda x: x < 26,
        "cdr": lambda x: x > 0,
        "cdrSum": lambda x: x > 0,
        "boston": lambda x: x < 15,
        "animal": lambda x: x < 15,
        "vege": lambda x: x < 15,
        "lm_imm": lambda x: x <= 8,
        "lm_del": lambda x: x <= 8,
        "trailA": lambda x: x > 78,
        "trailB": lambda x: x > 273,       
        "digitB": lambda x: x < 5,
        "digitF": lambda x: x < 5,
        "digitBL": lambda x: x < 5,
        "digitFL": lambda x: x < 5,
        "gds": lambda x: x >= 5
    }

    try:
        adni_df = pd.read_csv(ADNI_DATA_PATH)
    except FileNotFoundError:
        print(f"   ❌ 找不到患者数据 {ADNI_DATA_PATH}。")
        return

    patient_entities = {}
    valid_mapped_patients = 0
    total_features_mapped = 0

    print("\n   ⏳ 正在为每位患者建立固定长度的图谱实体序列与 Attention Mask...")
    
    for index, row in adni_df.iterrows():
        if 'filename' in row and pd.notna(row['filename']):
            patient_id = str(row['filename']).strip()
        else:
            continue

        active_nodes = []
        
        for col_name, val in row.items():
            if col_name in adni_to_pkg_map:
                if pd.notna(val) and str(val).strip().lower() not in ['', 'nan', '-4', '-1']:
                    try:
                        num_val = float(val)
                        rule = threshold_rules.get(col_name, lambda x: x > 0)
                        
                        if rule(num_val):
                            primekg_node = adni_to_pkg_map[col_name]
                            active_nodes.append(primekg_node)
                            total_features_mapped += 1
                    except ValueError:
                        pass

        active_nodes = list(set(active_nodes))
        seq_len = len(active_nodes)
        
        if seq_len > MAX_SEQ_LEN:
            active_nodes = active_nodes[:MAX_SEQ_LEN]
            attention_mask = [1] * MAX_SEQ_LEN
        else:
            pad_length = MAX_SEQ_LEN - seq_len
            attention_mask = [1] * seq_len + [0] * pad_length
            active_nodes = active_nodes + [PAD_TOKEN] * pad_length

        patient_entities[patient_id] = {
            "input_nodes": active_nodes,
            "attention_mask": attention_mask
        }
        
        if seq_len > 0:
            valid_mapped_patients += 1

    with open(OUTPUT_DICT_PATH, 'w', encoding='utf-8') as f:
        json.dump(patient_entities, f, indent=4, ensure_ascii=False)

    print("\n==================================================")
    print("💾 阶段 3 任务完成！")
    print(f"   👨‍⚕️ 处理患者总数: {len(adni_df)}")
    print(f"   🔗 成功为 {valid_mapped_patients} 名患者提取了图谱实体。")
    print(f"   📈 总计完成特征到实体的映射: {total_features_mapped} 次。")
    print(f"   📏 序列统一长度 (MAX_SEQ_LEN): {MAX_SEQ_LEN}")
    print(f"   📄 患者动态查询字典 (包含 Mask) 已保存至: {OUTPUT_DICT_PATH}")
    print("==================================================")

if __name__ == "__main__":
    build_patient_mapping()

In [ ]:
# Same DistMult score/loss; one shared full-graph pretraining job.
from clinicalKG.full_primekg import train_full_primekg, plot_loss
RESUME = False  # Set True to continue PrimeKG-DistMult-last.pt.
NUM_EPOCHS = 50  # Total target epochs, shared across cohorts.
EMBED_DIM = 128
BATCH_SIZE = 512
LR = 0.005
training_state = train_full_primekg(epochs=NUM_EPOCHS, embed_dim=EMBED_DIM,
                                   batch_size=BATCH_SIZE, lr=LR, resume=RESUME)
plot_loss()


In [ ]:
import matplotlib.pyplot as plt
import json
import os

# ================= 配置区 =================
LOSS_HISTORY_PATH = asset_path(DATASET, 'PrimeKG', 'ADNI-PrimeKG-Raw-LossHistory.json')
OUTPUT_PLOT_PATH = asset_path(DATASET, 'PrimeKG', 'ADNI-PrimeKG-Raw-LossCurve.png')

def plot_loss_curve():
    print("==================================================")
    print("📈 步骤 4: 正在绘制 KGE 训练 Loss 收敛曲线...")
    
    if not os.path.exists(LOSS_HISTORY_PATH):
        print(f"❌ 找不到 Loss 记录文件: {LOSS_HISTORY_PATH}，请确认上一步已成功训练。")
        return
        
    with open(LOSS_HISTORY_PATH, 'r', encoding='utf-8') as f:
        losses = json.load(f)
        
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(losses) + 1), losses, color='#2c3e50', linewidth=2, label='DistMult Train Loss')
    
    # 图表美化设置
    plt.title('DementiaHKG Knowledge Graph Embedding Loss', fontsize=16, fontweight='bold')
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Margin Ranking Loss', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=12)
    plt.tight_layout()
    
    # 保存高斯清晰度的图片供论文使用
    plt.savefig(OUTPUT_PLOT_PATH, dpi=300)
    print(f"   ✅ 曲线图已生成，正在当前 Cell 渲染，并已保存高清大图至: {OUTPUT_PLOT_PATH}")
    print("==================================================")
    
    plt.show()

if __name__ == "__main__":
    plot_loss_curve()